# YahooQuery Conventional Gap Research

Bu notebook production pipeline'dan veri okumaz ve ara çıktı kaydetmez. BIST universe filtrelerinden geçen hisselerin 5 dakikalık verisini YahooQuery'den doğrudan çeker, production conventional feature mantığını notebook içinde bar bazlı research akışında kullanır, gap-fill hipotezini inceler ve vectorbt ile backtest çalıştırır.

Ana fikirler:
- Common gap'lerin 5 dakikalık barlarda önceki bar high/low seviyesine kapanma eğilimini doğrula.
- Common Up sinyalinde short aç; çıkışı entry barında değil, sonraki barlarda önceki bar high hedefine dönüşte yap.
- Breakaway Up ve Runaway Up sinyallerinde long aç.
- Exhaustion Up sinyallerini short için daha düşük güvenli, validasyonlu sinyal olarak kullan.
- İşlem başına risk 100.000 TL sermayenin %1'ini aşmasın.
- Portföy ağırlıklarını train bölümünde efficient-frontier yaklaşımıyla optimize et.


## 1. Imports

In [255]:
from __future__ import annotations

from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import partial
from pathlib import Path
import sys
import time
import warnings

import numpy as np
import pandas as pd
import polars as pl
import requests
from yahooquery import Ticker

try:
    from tqdm.auto import tqdm
except ModuleNotFoundError:
    tqdm = None

try:
    import nbformat  # required by Plotly mime rendering in notebooks
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Plotly notebook rendering requires nbformat. Run `pip install nbformat` "
        "inside the active devcontainer virtualenv."
    ) from exc

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    import vectorbt as vbt
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "vectorbt is required for this notebook. Install it with `pip install vectorbt` "
        "or rebuild the devcontainer after the dependency update."
    ) from exc

try:
    from scipy.optimize import minimize
except Exception:
    minimize = None

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "kedro_project" / "src").exists():
        PROJECT_ROOT = candidate
        break

RUNTIME_KEDRO_SRC = Path("/opt/kedro_project/src")
WORKSPACE_KEDRO_SRC = PROJECT_ROOT / "kedro_project" / "src"
KEDRO_SRC = RUNTIME_KEDRO_SRC if RUNTIME_KEDRO_SRC.exists() else WORKSPACE_KEDRO_SRC

if not KEDRO_SRC.exists():
    raise FileNotFoundError(
        "Could not find Kedro source. Run `python scripts/bootstrap_runtime.py` "
        "from the project root, then reopen this notebook."
    )

if str(KEDRO_SRC) not in sys.path:
    sys.path.insert(0, str(KEDRO_SRC))

print(f"Using Kedro source: {KEDRO_SRC}")

from mlops_kedro.pipelines.stock_close_training.feature_engineering import (
    build_conventional_gap_trading_features,
)
from mlops_kedro.pipelines.stock_close_training.features.feature_engineering_utils import (
    add_model_training_tier_columns_for_symbol,
    map_by_symbol,
    prepare_feature_source,
    with_created_timestamp,
)
from mlops_kedro.pipelines.stock_close_training.features.indicators import (
    calculate_indicators_for_symbol,
)

pd.options.display.max_columns = 120
pd.options.display.float_format = "{:.4f}".format


def log_step(title: str, **values) -> None:
    print(f"\n[{title}]")
    for key, value in values.items():
        print(f"  {key}: {value}")


Using Kedro source: /opt/kedro_project/src


## 2. Research Parameters

In [256]:
SYMBOLS = []
USE_BIST_UNIVERSE = True
BIST_UNIVERSE_LIMIT = None
PLOT_SYMBOL_LIMIT = 5
EXCLUDE_DOMINANT_HOLDER_SYMBOLS = True
DOMINANT_HOLDER_THRESHOLD = 0.50
OWNERSHIP_FILTER_VERBOSE = True
OWNERSHIP_PROGRESS_EVERY = 25
OWNERSHIP_MAX_WORKERS = 12
PERIOD = "1y"
INTERVAL = "1d"
VECTORBT_FREQ = "1d"

INITIAL_CAPITAL = 100_000.0
MAX_RISK_FRACTION = 0.01
MAX_POSITION_RISK_TL = INITIAL_CAPITAL * MAX_RISK_FRACTION

TRAIN_FRACTION = 0.70
RISK_FREE_DAILY = 0.0
FEES = 0.001
SLIPPAGE = 0.001

ATR_WINDOW = 14
STOP_ATR_MULTIPLIER = 1.50
TAKE_PROFIT_R_MULTIPLIER = 2.50
GAP_LEVERAGE_BUCKET = 0.08
MAX_INTEGER_LEVERAGE = 3
MIN_GAP_ABS = 0.0025
EFFICIENT_FRONTIER_TRIALS = 20_000

TIME_ENCODING_CONFIG = {
    "close_model_freq": "B",
    "source_harmonics": [1, 2],
    "close_model_harmonics": [1],
    "periods": {
        "month": 12.0,
        "day": 31.0,
        "day_of_year": 366.0,
    },
}

PRICE_COLUMNS = ["open", "high", "low", "close", "volume"]
CONDITION_COLUMNS = [
    "gap_up",
    "gap_down",
    "exhaustion_gap_up",
    "exhaustion_gap_down",
    "high_vol",
    "extreme_vol",
    "breakout_up",
    "breakout_down",
    "consolidation",
    "uptrend",
    "downtrend",
    "overbought",
    "oversold",
]
STRATEGY_REQUIRED_CONDITIONS = [
    "gap_up",
    "gap_down",
    "breakout_up",
    "breakout_down",
    "high_vol",
    "extreme_vol",
    "consolidation",
    "uptrend",
    "downtrend",
    "overbought",
    "oversold",
]
MODEL_TIME_FEATURES = [
    "calendar_gap_days",
    "month_sin_1",
    "month_cos_1",
    "month_sin_2",
    "month_cos_2",
    "day_sin_1",
    "day_cos_1",
    "day_sin_2",
    "day_cos_2",
    "day_of_year_sin_1",
    "day_of_year_cos_1",
    "day_of_year_sin_2",
    "day_of_year_cos_2",
]
INDICATOR_COLUMNS = [
    "RSI",
    "SMA_Short",
    "SMA_Long",
    "Vol_SMA",
    "Range_high",
    "Range_low",
    "ADX",
]
INDICATOR_FEATURE_COLUMNS = [
    "symbol",
    "date",
    "created_timestamp",
    *PRICE_COLUMNS,
    "target_close",
    "prev_open",
    "prev_high",
    "prev_low",
    "prev_volume",
    *MODEL_TIME_FEATURES,
    *INDICATOR_COLUMNS,
]
CONVENTIONAL_GAP_TRADING_COLUMNS = [
    *INDICATOR_FEATURE_COLUMNS,
    *CONDITION_COLUMNS,
    "Gap_Type",
]
COLUMNS_CONFIG = {
    "indicator_features": INDICATOR_FEATURE_COLUMNS,
    "condition": CONDITION_COLUMNS,
    "strategy_required_conditions": STRATEGY_REQUIRED_CONDITIONS,
    "conventional_gap_trading": CONVENTIONAL_GAP_TRADING_COLUMNS,
}


log_step(
    "research_config",
    use_bist_universe=USE_BIST_UNIVERSE,
    bist_universe_limit=BIST_UNIVERSE_LIMIT,
    plot_symbol_limit=PLOT_SYMBOL_LIMIT,
    exclude_dominant_holder_symbols=EXCLUDE_DOMINANT_HOLDER_SYMBOLS,
    dominant_holder_threshold=DOMINANT_HOLDER_THRESHOLD,
    ownership_filter_verbose=OWNERSHIP_FILTER_VERBOSE,
    ownership_progress_every=OWNERSHIP_PROGRESS_EVERY,
    ownership_max_workers=OWNERSHIP_MAX_WORKERS,
    period=PERIOD,
    interval=INTERVAL,
    vectorbt_freq=VECTORBT_FREQ,
    max_risk_fraction=MAX_RISK_FRACTION,
    efficient_frontier_trials=EFFICIENT_FRONTIER_TRIALS,
)



[research_config]
  use_bist_universe: True
  bist_universe_limit: None
  plot_symbol_limit: 5
  exclude_dominant_holder_symbols: True
  dominant_holder_threshold: 0.5
  ownership_filter_verbose: True
  ownership_progress_every: 25
  ownership_max_workers: 12
  period: 1y
  interval: 1d
  vectorbt_freq: 1d
  max_risk_fraction: 0.01
  efficient_frontier_trials: 20000


## 3. BIST Universe And Liquidity Filters


In [257]:
def progress_write(message: str) -> None:
    if tqdm is not None:
        tqdm.write(message)
    else:
        print(message)


def get_bist_symbols():
    url = "https://scanner.tradingview.com/turkey/scan"
    payload = {
        "columns": ["name"],
        "filter": [
            {"left": "exchange", "operation": "equal", "right": "BIST"},
            {"left": "type", "operation": "equal", "right": "stock"},
        ],
        "range": [0, 1000],
    }
    started_at = time.perf_counter()
    print("[get_bist_symbols] requesting TradingView BIST scan...")
    response = requests.post(url, json=payload, timeout=30)
    response.raise_for_status()
    data = response.json()
    symbols = [item["d"][0] for item in data["data"]]
    log_step(
        "tradingview_scan_complete",
        status_code=response.status_code,
        elapsed_seconds=round(time.perf_counter() - started_at, 2),
        raw_symbol_count=len(symbols),
        sample=symbols[:10],
    )
    return symbols


def to_yahoo_bist_symbol(symbol: str) -> str:
    symbol = str(symbol).strip().upper()
    if not symbol:
        return symbol
    return symbol if "." in symbol else f"{symbol}.IS"


def normalize_yahoo_symbols(symbols: list[str]) -> list[str]:
    return sorted({to_yahoo_bist_symbol(symbol) for symbol in symbols if str(symbol).strip()})


def payload_records(payload, symbol: str, source: str) -> list[dict]:
    if payload is None:
        return []
    if isinstance(payload, pd.DataFrame):
        frame = payload.reset_index()
        if "symbol" not in frame.columns:
            frame["symbol"] = symbol
        frame["source"] = source
        return frame.to_dict("records")
    if isinstance(payload, pd.Series):
        row = payload.to_dict()
        row.update({"symbol": symbol, "source": source})
        return [row]
    if isinstance(payload, list):
        rows = []
        for item in payload:
            rows.extend(payload_records(item, symbol, source))
        return rows
    if isinstance(payload, dict):
        if symbol in payload:
            return payload_records(payload[symbol], symbol, source)
        nested_rows = []
        for key, value in payload.items():
            if isinstance(value, (dict, list, pd.DataFrame, pd.Series)):
                nested_rows.extend(payload_records(value, symbol, f"{source}.{key}"))
        if nested_rows:
            return nested_rows
        row = dict(payload)
        row.update({"symbol": symbol, "source": source})
        return [row]
    return []


def coerce_holder_percent(value) -> float:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan
    if isinstance(value, str):
        cleaned = value.strip().replace("%", "").replace(",", ".")
        if not cleaned:
            return np.nan
        value = cleaned
    try:
        pct = float(value)
    except (TypeError, ValueError):
        return np.nan
    return pct / 100.0 if pct > 1.0 else pct


def fetch_holder_snapshot(symbol: str, verbose: bool = False) -> pd.DataFrame:
    ticker = Ticker(symbol, asynchronous=False)
    rows = []
    source_timings = {}
    for source in ["institution_ownership", "fund_ownership", "major_holders"]:
        source_started_at = time.perf_counter()
        try:
            payload = getattr(ticker, source)
            source_rows = payload_records(payload, symbol, source)
            rows.extend(source_rows)
            source_timings[source] = {
                "rows": len(source_rows),
                "elapsed_s": round(time.perf_counter() - source_started_at, 2),
            }
        except Exception as exc:
            elapsed_s = round(time.perf_counter() - source_started_at, 2)
            source_timings[source] = {"rows": 0, "elapsed_s": elapsed_s, "error": str(exc)[:120]}
            rows.append({"symbol": symbol, "source": source, "holder_error": str(exc)})

    if verbose:
        progress_write(f"  [ownership_sources] {symbol} -> {source_timings}")

    if not rows:
        return pd.DataFrame(columns=["symbol", "source", "holder_name", "holder_pct"])

    frame = pd.DataFrame(rows)
    name_columns = ["organization", "holder", "Holder", "name", "Name"]
    percent_columns = ["pctHeld", "pct_held", "percentHeld", "Percent Held", "% Out", "% Shares Out", "sharesPercentSharesOut"]

    frame["holder_name"] = np.nan
    for column in name_columns:
        if column in frame.columns:
            frame["holder_name"] = frame["holder_name"].combine_first(frame[column])

    frame["holder_pct"] = np.nan
    for column in percent_columns:
        if column in frame.columns:
            frame["holder_pct"] = frame["holder_pct"].combine_first(frame[column].map(coerce_holder_percent))

    return frame


def evaluate_symbol_dominant_holder(
    index: int,
    symbol: str,
    threshold: float,
) -> dict:
    symbol_started_at = time.perf_counter()
    try:
        holder_rows = fetch_holder_snapshot(symbol, verbose=False)
        error_count = int(holder_rows["holder_error"].notna().sum()) if "holder_error" in holder_rows.columns else 0
        named_holders = holder_rows.dropna(subset=["holder_name", "holder_pct"]).copy()
        named_holders = named_holders[named_holders["holder_name"].astype(str).str.strip().ne("")]
        elapsed_s = round(time.perf_counter() - symbol_started_at, 2)

        if named_holders.empty:
            row = {
                "input_order": index,
                "symbol": symbol,
                "kept": True,
                "status": "kept_no_named_holder_data",
                "dominant_holder": None,
                "dominant_holder_pct": np.nan,
                "source": None,
                "raw_holder_rows": len(holder_rows),
                "named_holder_rows": 0,
                "elapsed_seconds": elapsed_s,
            }
            return {
                "row": row,
                "errors": error_count,
                "no_named_holder_data": 1,
                "status_message": f"kept(no holder data) elapsed={elapsed_s}s",
            }

        dominant = named_holders.sort_values("holder_pct", ascending=False).iloc[0]
        dominant_pct = float(dominant["holder_pct"])
        excluded = dominant_pct > threshold
        row = {
            "input_order": index,
            "symbol": symbol,
            "kept": not excluded,
            "status": "excluded_dominant_holder" if excluded else "kept",
            "dominant_holder": dominant["holder_name"],
            "dominant_holder_pct": dominant_pct,
            "source": dominant.get("source"),
            "raw_holder_rows": len(holder_rows),
            "named_holder_rows": len(named_holders),
            "elapsed_seconds": elapsed_s,
        }
        return {
            "row": row,
            "errors": error_count,
            "no_named_holder_data": 0,
            "status_message": (
                f"{'excluded' if excluded else 'kept'} "
                f"holder={dominant['holder_name']} pct={dominant_pct:.1%} elapsed={elapsed_s}s"
            ),
        }
    except Exception as exc:
        elapsed_s = round(time.perf_counter() - symbol_started_at, 2)
        row = {
            "input_order": index,
            "symbol": symbol,
            "kept": True,
            "status": "kept_holder_fetch_error",
            "dominant_holder": None,
            "dominant_holder_pct": np.nan,
            "source": None,
            "raw_holder_rows": 0,
            "named_holder_rows": 0,
            "elapsed_seconds": elapsed_s,
            "error": str(exc)[:300],
        }
        return {
            "row": row,
            "errors": 1,
            "no_named_holder_data": 1,
            "status_message": f"kept(fetch error) error={str(exc)[:120]} elapsed={elapsed_s}s",
        }


def filter_symbols_by_dominant_holder(
    symbols: list[str],
    threshold: float = DOMINANT_HOLDER_THRESHOLD,
    verbose: bool = OWNERSHIP_FILTER_VERBOSE,
    print_every: int = OWNERSHIP_PROGRESS_EVERY,
    max_workers: int = OWNERSHIP_MAX_WORKERS,
) -> tuple[list[str], pd.DataFrame]:
    report_rows = []
    counters = {"kept": 0, "excluded": 0, "no_named_holder_data": 0, "errors": 0}
    started_at = time.perf_counter()
    max_workers = max(1, min(int(max_workers), len(symbols) or 1))

    log_step(
        "ownership_filter_start",
        symbol_count=len(symbols),
        threshold=threshold,
        verbose=verbose,
        print_every=print_every,
        max_workers=max_workers,
    )

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(evaluate_symbol_dominant_holder, index, symbol, threshold)
            for index, symbol in enumerate(symbols, start=1)
        ]
        completed_futures = as_completed(futures)
        if tqdm is not None:
            completed_futures = tqdm(
                completed_futures,
                total=len(futures),
                desc=f"ownership filter x{max_workers}",
                unit="symbol",
            )

        for completed_count, future in enumerate(completed_futures, start=1):
            result = future.result()
            row = result["row"]
            report_rows.append(row)

            counters["kept"] += int(bool(row["kept"]))
            counters["excluded"] += int(row["status"] == "excluded_dominant_holder")
            counters["no_named_holder_data"] += int(result["no_named_holder_data"])
            counters["errors"] += int(result["errors"])

            if tqdm is not None and hasattr(completed_futures, "set_postfix"):
                completed_futures.set_postfix(
                    kept=counters["kept"],
                    excluded=counters["excluded"],
                    no_data=counters["no_named_holder_data"],
                    errors=counters["errors"],
                )

            should_print = verbose and (
                completed_count == 1
                or completed_count == len(symbols)
                or completed_count % max(1, print_every) == 0
                or row["status"] in {"excluded_dominant_holder", "kept_holder_fetch_error"}
            )
            if should_print:
                progress_write(
                    f"  [{completed_count}/{len(symbols)} done] {row['symbol']}: {result['status_message']} | "
                    f"kept={counters['kept']} excluded={counters['excluded']} no_data={counters['no_named_holder_data']} errors={counters['errors']}"
                )

    report = pd.DataFrame(report_rows).sort_values("input_order").reset_index(drop=True)
    kept_symbols = report.loc[report["kept"], "symbol"].to_list()

    log_step(
        "ownership_filter_complete",
        elapsed_seconds=round(time.perf_counter() - started_at, 2),
        max_workers=max_workers,
        kept_symbol_count=len(kept_symbols),
        excluded_symbol_count=counters["excluded"],
        no_holder_data_count=counters["no_named_holder_data"],
        error_count=counters["errors"],
    )
    return kept_symbols, report


log_step(
    "symbol_universe_start",
    use_bist_universe=USE_BIST_UNIVERSE,
    input_symbols=SYMBOLS if SYMBOLS else "from_tradingview",
)

if USE_BIST_UNIVERSE:
    raw_bist_symbols = get_bist_symbols()
    log_step("tradingview_bist_scan", raw_symbol_count=len(raw_bist_symbols), sample=raw_bist_symbols[:10])
    SYMBOLS = normalize_yahoo_symbols(raw_bist_symbols)
    if BIST_UNIVERSE_LIMIT is not None:
        SYMBOLS = SYMBOLS[: int(BIST_UNIVERSE_LIMIT)]
        print(f"  applied BIST_UNIVERSE_LIMIT -> {len(SYMBOLS)} symbols")
else:
    SYMBOLS = normalize_yahoo_symbols(SYMBOLS)

log_step("normalized_yahoo_symbols", symbol_count=len(SYMBOLS), sample=SYMBOLS[:10])

if EXCLUDE_DOMINANT_HOLDER_SYMBOLS:
    SYMBOLS, holder_report = filter_symbols_by_dominant_holder(SYMBOLS)
else:
    holder_report = pd.DataFrame({"symbol": SYMBOLS, "status": "ownership_filter_disabled"})

display(holder_report)
log_step(
    "ownership_filter_result",
    kept_symbol_count=len(SYMBOLS),
    excluded_symbol_count=int(holder_report["status"].eq("excluded_dominant_holder").sum()) if "status" in holder_report.columns else 0,
    kept_sample=SYMBOLS[:10],
)
print(f"Selected symbols ({len(SYMBOLS)}): {SYMBOLS}")
if not SYMBOLS:
    raise ValueError("No symbols left after BIST universe / ownership filters.")



[symbol_universe_start]
  use_bist_universe: True
  input_symbols: from_tradingview
[get_bist_symbols] requesting TradingView BIST scan...



[tradingview_scan_complete]
  status_code: 200
  elapsed_seconds: 0.34
  raw_symbol_count: 613
  sample: ['ANSGR', 'OYLUM', 'KATMR', 'HURGZ', 'SILVR', 'ARENA', 'ERBOS', 'GARAN', 'ISKUR', 'TURSG']

[tradingview_bist_scan]
  raw_symbol_count: 613
  sample: ['ANSGR', 'OYLUM', 'KATMR', 'HURGZ', 'SILVR', 'ARENA', 'ERBOS', 'GARAN', 'ISKUR', 'TURSG']

[normalized_yahoo_symbols]
  symbol_count: 613
  sample: ['A1CAP.IS', 'A1YEN.IS', 'AAGYO.IS', 'ACSEL.IS', 'ADEL.IS', 'ADESE.IS', 'ADGYO.IS', 'AEFES.IS', 'AFYON.IS', 'AGESA.IS']

[ownership_filter_start]
  symbol_count: 613
  threshold: 0.5
  verbose: True
  print_every: 25
  max_workers: 12


ownership filter x12:   0%|          | 0/613 [00:00<?, ?symbol/s]

  [1/613 done] AGHOL.IS: kept(no holder data) elapsed=1.68s | kept=1 excluded=0 no_data=1 errors=0
  [25/613 done] AKSA.IS: kept(no holder data) elapsed=2.04s | kept=25 excluded=0 no_data=24 errors=0
  [50/613 done] ARSAN.IS: kept(no holder data) elapsed=1.81s | kept=50 excluded=0 no_data=48 errors=0
  [75/613 done] AZTEK.IS: kept(no holder data) elapsed=2.33s | kept=75 excluded=0 no_data=73 errors=0
  [100/613 done] BLUME.IS: kept(no holder data) elapsed=1.68s | kept=100 excluded=0 no_data=97 errors=0
  [125/613 done] BRYAT.IS: kept(no holder data) elapsed=4.46s | kept=125 excluded=0 no_data=122 errors=0
  [150/613 done] DERIM.IS: kept(no holder data) elapsed=1.62s | kept=150 excluded=0 no_data=146 errors=0
  [175/613 done] DURKN.IS: kept(no holder data) elapsed=1.65s | kept=175 excluded=0 no_data=171 errors=0
  [200/613 done] EKSUN.IS: kept(no holder data) elapsed=2.68s | kept=200 excluded=0 no_data=196 errors=0
  [225/613 done] FMIZP.IS: kept(no holder data) elapsed=1.64s | kept=225

,input_order,symbol,kept,status,dominant_holder,dominant_holder_pct,source,raw_holder_rows,named_holder_rows,elapsed_seconds,error
0,1,A1CAP.IS,True,kept_no_named_holder_data,NaN,NaN,NaN,1,0,2.0100,NaN
1,2,A1YEN.IS,True,kept_no_named_holder_data,NaN,NaN,NaN,1,0,2.3900,NaN
2,3,AAGYO.IS,True,kept_no_named_holder_data,NaN,NaN,NaN,1,0,1.9100,NaN
3,4,ACSEL.IS,True,kept_no_named_holder_data,NaN,NaN,NaN,1,0,2.1700,NaN
4,5,ADEL.IS,True,kept_no_named_holder_data,NaN,NaN,NaN,1,0,2.2400,NaN
...,...,...,...,...,...,...,...,...,...,...,...
608,609,ZEDUR.IS,True,kept_no_named_holder_data,NaN,NaN,NaN,0,0,2.3900,NaN
609,610,ZERGY.IS,True,kept_no_named_holder_data,NaN,NaN,NaN,0,0,3.4400,NaN
610,611,ZGYO.IS,True,kept_no_named_holder_data,NaN,NaN,NaN,0,0,2.7300,NaN
611,612,ZOREN.IS,True,kept_no_named_holder_data,NaN,NaN,NaN,0,0,1.5300,NaN



[ownership_filter_result]
  kept_symbol_count: 613
  excluded_symbol_count: 0
  kept_sample: ['A1CAP.IS', 'A1YEN.IS', 'AAGYO.IS', 'ACSEL.IS', 'ADEL.IS', 'ADESE.IS', 'ADGYO.IS', 'AEFES.IS', 'AFYON.IS', 'AGESA.IS']
Selected symbols (613): ['A1CAP.IS', 'A1YEN.IS', 'AAGYO.IS', 'ACSEL.IS', 'ADEL.IS', 'ADESE.IS', 'ADGYO.IS', 'AEFES.IS', 'AFYON.IS', 'AGESA.IS', 'AGHOL.IS', 'AGROT.IS', 'AGYO.IS', 'AHGAZ.IS', 'AHSGY.IS', 'AKBNK.IS', 'AKCNS.IS', 'AKENR.IS', 'AKFGY.IS', 'AKFIS.IS', 'AKFYE.IS', 'AKGRT.IS', 'AKHAN.IS', 'AKMGY.IS', 'AKSA.IS', 'AKSEN.IS', 'AKSGY.IS', 'AKSUE.IS', 'AKYHO.IS', 'ALARK.IS', 'ALBRK.IS', 'ALCAR.IS', 'ALCTL.IS', 'ALFAS.IS', 'ALGYO.IS', 'ALKA.IS', 'ALKIM.IS', 'ALKLC.IS', 'ALTIN.IS', 'ALTNY.IS', 'ALVES.IS', 'ANELE.IS', 'ANGEN.IS', 'ANHYT.IS', 'ANSGR.IS', 'ARASE.IS', 'ARCLK.IS', 'ARDYZ.IS', 'ARENA.IS', 'ARFYE.IS', 'ARMGD.IS', 'ARSAN.IS', 'ARTMS.IS', 'ARZUM.IS', 'ASELS.IS', 'ASGYO.IS', 'ASTOR.IS', 'ASUZU.IS', 'ATAGY.IS', 'ATAKP.IS', 'ATATP.IS', 'ATATR.IS', 'ATEKS.IS', 'ATLAS.IS

## 4. Pull 5-Minute Data From YahooQuery


In [258]:
log_step("yahoo_request", symbol_count=len(SYMBOLS), period=PERIOD, interval=INTERVAL, sample=SYMBOLS[:10])
raw_history = Ticker(SYMBOLS, asynchronous=False).history(period=PERIOD, interval=INTERVAL)

if raw_history is None or len(raw_history) == 0:
    raise RuntimeError("YahooQuery returned no rows for the requested symbols.")

prices_pd = raw_history.reset_index()
prices_pd = prices_pd.rename(columns={"level_0": "symbol"})
if "date" not in prices_pd.columns and "index" in prices_pd.columns:
    prices_pd = prices_pd.rename(columns={"index": "date"})

required = ["symbol", "date", "open", "high", "low", "close", "volume"]
missing = [column for column in required if column not in prices_pd.columns]
if missing:
    raise ValueError(f"YahooQuery output is missing columns: {missing}. Columns={list(prices_pd.columns)}")

prices_pd = prices_pd[required].copy()
prices_pd["date"] = pd.to_datetime(prices_pd["date"]).dt.tz_localize(None)
log_step(
    "yahoo_raw_result",
    rows=len(prices_pd),
    returned_symbol_count=prices_pd["symbol"].nunique(),
    min_date=prices_pd["date"].min(),
    max_date=prices_pd["date"].max(),
)

available_symbols = sorted(prices_pd["symbol"].dropna().unique())
missing_symbols = sorted(set(SYMBOLS).difference(available_symbols))
if missing_symbols:
    print(f"YahooQuery returned no rows for these symbols and they will be removed: {missing_symbols}")

completeness_report = (
    prices_pd.groupby("symbol", observed=True)
    .agg(
        rows=("date", "size"),
        duplicate_timestamps=("date", lambda values: int(values.duplicated().sum())),
        **{f"{column}_missing": (column, lambda values: int(values.isna().sum())) for column in required},
    )
    .reset_index()
)
missing_columns = [column for column in completeness_report.columns if column.endswith("_missing")]
incomplete_mask = completeness_report[missing_columns].sum(axis=1).gt(0) | completeness_report["duplicate_timestamps"].gt(0)
incomplete_symbols = completeness_report.loc[incomplete_mask, "symbol"].to_list()

display(completeness_report)
log_step(
    "completeness_check",
    checked_symbols=len(completeness_report),
    incomplete_symbol_count=len(incomplete_symbols),
    incomplete_symbols=incomplete_symbols[:20],
)
if incomplete_symbols:
    print(f"Symbols removed because OHLCV/date columns are incomplete or duplicated: {incomplete_symbols}")
    prices_pd = prices_pd[~prices_pd["symbol"].isin(incomplete_symbols)].copy()

prices_pd = prices_pd.dropna(subset=required).sort_values(["symbol", "date"]).reset_index(drop=True)
SYMBOLS = sorted(prices_pd["symbol"].unique())
if not SYMBOLS:
    raise ValueError("No symbols left after YahooQuery completeness validation.")

prices_pl = pl.from_pandas(prices_pd)

summary = (
    prices_pd.groupby("symbol")
    .agg(
        rows=("date", "size"),
        min_date=("date", "min"),
        max_date=("date", "max"),
        last_close=("close", "last"),
        avg_volume=("volume", "mean"),
    )
    .reset_index()
)
plot_symbol_report = summary.sort_values(["rows", "avg_volume"], ascending=[False, False]).head(PLOT_SYMBOL_LIMIT)
PLOT_SYMBOLS = plot_symbol_report["symbol"].to_list()

display(summary)
log_step(
    "validated_price_data",
    final_symbol_count=len(SYMBOLS),
    final_rows=len(prices_pd),
    selected_plot_symbols=PLOT_SYMBOLS,
)
print(f"Plot symbols ({len(PLOT_SYMBOLS)} of {len(SYMBOLS)}): {PLOT_SYMBOLS}")
prices_pd.tail()



[yahoo_request]
  symbol_count: 613
  period: 1y
  interval: 1d
  sample: ['A1CAP.IS', 'A1YEN.IS', 'AAGYO.IS', 'ACSEL.IS', 'ADEL.IS', 'ADESE.IS', 'ADGYO.IS', 'AEFES.IS', 'AFYON.IS', 'AGESA.IS']

[yahoo_raw_result]
  rows: 150385
  returned_symbol_count: 611
  min_date: 2025-07-14 00:00:00
  max_date: 2026-07-13 00:00:00
YahooQuery returned no rows for these symbols and they will be removed: ['ALTIN.IS', 'DMLKT.IS']


,symbol,rows,duplicate_timestamps,symbol_missing,date_missing,open_missing,high_missing,low_missing,close_missing,volume_missing
0,A1CAP.IS,254,0,0,0,0,0,0,0,0
1,A1YEN.IS,256,0,0,0,0,0,0,0,0
2,AAGYO.IS,65,0,0,0,0,0,0,0,0
3,ACSEL.IS,254,0,0,0,0,0,0,0,0
4,ADEL.IS,254,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
606,ZEDUR.IS,254,0,0,0,0,0,0,0,0
607,ZERGY.IS,143,0,0,0,0,0,0,0,0
608,ZGYO.IS,123,0,0,0,0,0,0,0,0
609,ZOREN.IS,254,0,0,0,0,0,0,0,0



[completeness_check]
  checked_symbols: 611
  incomplete_symbol_count: 1
  incomplete_symbols: ['ISKUR.IS']
Symbols removed because OHLCV/date columns are incomplete or duplicated: ['ISKUR.IS']


,symbol,rows,min_date,max_date,last_close,avg_volume
0,A1CAP.IS,254,2025-07-14,2026-07-13,9.9600,19951585.2559
1,A1YEN.IS,256,2025-07-14,2026-07-13,2.9000,31033757.9180
2,AAGYO.IS,65,2026-04-09,2026-07-13,14.8900,36804191.8769
3,ACSEL.IS,254,2025-07-14,2026-07-13,129.5000,337857.9764
4,ADEL.IS,254,2025-07-14,2026-07-13,30.0200,4177385.8858
...,...,...,...,...,...,...
605,ZEDUR.IS,254,2025-07-14,2026-07-13,8.6700,3752375.7165
606,ZERGY.IS,143,2025-12-18,2026-07-13,10.2600,36615564.6993
607,ZGYO.IS,123,2026-01-16,2026-07-13,38.5800,16115478.0000
608,ZOREN.IS,254,2025-07-14,2026-07-13,2.6700,73723855.3465



[validated_price_data]
  final_symbol_count: 610
  final_rows: 150338
  selected_plot_symbols: ['TURSG.IS', 'HLGYO.IS', 'ENTRA.IS', 'OSTIM.IS', 'GOODY.IS']
Plot symbols (5 of 610): ['TURSG.IS', 'HLGYO.IS', 'ENTRA.IS', 'OSTIM.IS', 'GOODY.IS']


,symbol,date,open,high,low,close,volume
150333,ZRGYO.IS,2026-07-07,17.8000,17.8000,17.4200,17.4200,1184367.0000
150334,ZRGYO.IS,2026-07-08,17.4200,17.6900,17.0000,17.5000,2263469.0000
150335,ZRGYO.IS,2026-07-09,17.5300,17.8900,17.5000,17.7200,1072686.0000
150336,ZRGYO.IS,2026-07-10,17.7200,18.3200,17.6700,18.0500,4061994.0000
150337,ZRGYO.IS,2026-07-13,18.0500,18.0500,17.5200,17.6400,681750.0000


## 4. Production Conventional Features In-Memory

Burada conventional feature tarafında yeni fonksiyon yazılmıyor; production script fonksiyonları doğrudan kullanılıyor.

In [259]:
log_step("feature_engineering_start", input_rows=len(prices_pd), input_symbols=len(SYMBOLS))
feature_source = prepare_feature_source(prices_pl, TIME_ENCODING_CONFIG)
log_step("feature_source_ready", rows=feature_source.height, columns=len(feature_source.columns))
feature_enriched_prices = map_by_symbol(
    feature_source.drop_nulls(["symbol", "date"]).sort(["symbol", "date"]),
    partial(
        add_model_training_tier_columns_for_symbol,
        time_encoding_config=TIME_ENCODING_CONFIG,
    ),
)
log_step("tier_columns_ready", rows=feature_enriched_prices.height, columns=len(feature_enriched_prices.columns))
indicator_features = map_by_symbol(
    feature_enriched_prices,
    calculate_indicators_for_symbol,
)
log_step("indicator_raw_ready", rows=indicator_features.height, columns=len(indicator_features.columns))
indicator_features = with_created_timestamp(indicator_features).select(
    COLUMNS_CONFIG["indicator_features"]
)

required_indicator_columns = {"symbol", "date", "open", "high", "low", "close", "volume"}
missing_indicator_columns = required_indicator_columns.difference(indicator_features.columns)
if missing_indicator_columns:
    raise ValueError(
        "Indicator features are missing required conventional columns: "
        f"{sorted(missing_indicator_columns)}. Current columns={indicator_features.columns}"
    )
if indicator_features.is_empty():
    raise ValueError(
        "Indicator features are empty after YahooQuery preprocessing. "
        "Check the previous cell's YahooQuery summary output."
    )

conventional_features = build_conventional_gap_trading_features(
    indicator_features,
    COLUMNS_CONFIG,
)

log_step("conventional_features_ready", rows=conventional_features.height, columns=len(conventional_features.columns))
features = conventional_features.to_pandas()
features["date"] = pd.to_datetime(features["date"]).dt.tz_localize(None)
features = features.sort_values(["symbol", "date"]).reset_index(drop=True)

features["prev_close_research"] = features.groupby("symbol", observed=True)["close"].shift(1)
features["prev_high_research"] = features.groupby("symbol", observed=True)["high"].shift(1)
features["prev_low_research"] = features.groupby("symbol", observed=True)["low"].shift(1)
features["gap_pct"] = features["open"] / features["prev_close_research"] - 1.0
features["common_gap_fill_target"] = np.select(
    [
        features["Gap_Type"].eq("Common Up"),
        features["Gap_Type"].eq("Common Down"),
    ],
    [
        features["prev_high_research"],
        features["prev_low_research"],
    ],
    default=np.nan,
)

true_range_parts = pd.concat(
    [
        (features["high"] - features["low"]).abs(),
        (features["high"] - features["prev_close_research"]).abs(),
        (features["low"] - features["prev_close_research"]).abs(),
    ],
    axis=1,
)
features["true_range"] = true_range_parts.max(axis=1)
features["ATR"] = (
    features.groupby("symbol", observed=True)["true_range"]
    .transform(lambda series: series.rolling(ATR_WINDOW, min_periods=3).mean())
)

signal_counts = (
    features.groupby(["symbol", "Gap_Type"], observed=True)
    .size()
    .rename("rows")
    .reset_index()
    .sort_values(["symbol", "rows"], ascending=[True, False])
)
display(signal_counts)
log_step(
    "gap_signal_counts",
    total_rows=len(features),
    total_non_none_signals=int(features["Gap_Type"].ne("None").sum()),
    top_signal_rows=signal_counts.head(10).to_dict("records"),
)
features.tail()



[feature_engineering_start]
  input_rows: 150338
  input_symbols: 610

[feature_source_ready]
  rows: 150338
  columns: 13

[tier_columns_ready]
  rows: 150338
  columns: 28

[indicator_raw_ready]
  rows: 150338
  columns: 35

[conventional_features_ready]
  rows: 150338
  columns: 47


,symbol,Gap_Type,rows
4,A1CAP.IS,None,241
1,A1CAP.IS,Common Down,5
2,A1CAP.IS,Common Up,3
0,A1CAP.IS,Breakaway Down,2
5,A1CAP.IS,Runaway Up,2
...,...,...,...
2594,ZOREN.IS,Runaway Up,1
2596,ZRGYO.IS,None,250
2595,ZRGYO.IS,Common Down,2
2597,ZRGYO.IS,Runaway Down,1



[gap_signal_counts]
  total_rows: 150338
  total_non_none_signals: 5963
  top_signal_rows: [{'symbol': 'A1CAP.IS', 'Gap_Type': 'None', 'rows': 241}, {'symbol': 'A1CAP.IS', 'Gap_Type': 'Common Down', 'rows': 5}, {'symbol': 'A1CAP.IS', 'Gap_Type': 'Common Up', 'rows': 3}, {'symbol': 'A1CAP.IS', 'Gap_Type': 'Breakaway Down', 'rows': 2}, {'symbol': 'A1CAP.IS', 'Gap_Type': 'Runaway Up', 'rows': 2}, {'symbol': 'A1CAP.IS', 'Gap_Type': 'Exhaustion Up', 'rows': 1}, {'symbol': 'A1YEN.IS', 'Gap_Type': 'None', 'rows': 246}, {'symbol': 'A1YEN.IS', 'Gap_Type': 'Common Down', 'rows': 3}, {'symbol': 'A1YEN.IS', 'Gap_Type': 'Common Up', 'rows': 2}, {'symbol': 'A1YEN.IS', 'Gap_Type': 'Exhaustion Up', 'rows': 2}]


,symbol,date,created_timestamp,open,high,low,close,volume,target_close,prev_open,prev_high,prev_low,prev_volume,calendar_gap_days,month_sin_1,month_cos_1,month_sin_2,month_cos_2,day_sin_1,day_cos_1,day_sin_2,day_cos_2,day_of_year_sin_1,day_of_year_cos_1,day_of_year_sin_2,day_of_year_cos_2,RSI,SMA_Short,SMA_Long,Vol_SMA,Range_high,Range_low,ADX,gap_up,gap_down,exhaustion_gap_up,exhaustion_gap_down,high_vol,extreme_vol,breakout_up,breakout_down,consolidation,uptrend,downtrend,overbought,oversold,Gap_Type,prev_close_research,prev_high_research,prev_low_research,gap_pct,common_gap_fill_target,true_range,ATR
150333,ZRGYO.IS,2026-07-07,2026-07-13 11:30:42.704304+00:00,17.8000,17.8000,17.4200,17.4200,1184367.0000,17.4200,17.8500,18.3000,17.8000,3162962.0000,0,-0.5000,-0.8660,0.8660,0.5000,0.9885,0.1514,0.2994,-0.9541,-0.0857,-0.9963,0.1708,0.9853,44.2978,17.5920,18.5668,17084673.9000,18.4700,16.5000,18.2364,False,False,False,True,False,False,False,False,True,False,True,False,False,None,17.8200,18.3000,17.8000,-0.0011,NaN,0.4000,0.5414
150334,ZRGYO.IS,2026-07-08,2026-07-13 11:30:42.704304+00:00,17.4200,17.6900,17.0000,17.5000,2263469.0000,17.5000,17.8000,17.8000,17.4200,1184367.0000,0,-0.5000,-0.8660,0.8660,0.5000,0.9987,-0.0506,-0.1012,-0.9949,-0.1028,-0.9947,0.2046,0.9789,45.5012,17.6085,18.4656,16909352.9000,18.4700,16.5000,18.4046,False,False,False,False,False,False,False,False,True,False,True,False,False,None,17.4200,17.8000,17.4200,0.0000,NaN,0.6900,0.5521
150335,ZRGYO.IS,2026-07-09,2026-07-13 11:30:42.704304+00:00,17.5300,17.8900,17.5000,17.7200,1072686.0000,17.7200,17.4200,17.6900,17.0000,2263469.0000,0,-0.5000,-0.8660,0.8660,0.5000,0.9681,-0.2507,-0.4853,-0.8743,-0.1199,-0.9928,0.2380,0.9713,48.7783,17.6495,18.3888,16592560.9500,18.4700,16.5000,18.1140,False,False,True,False,False,False,False,False,True,False,True,False,False,None,17.5000,17.6900,17.0000,0.0017,NaN,0.3900,0.5550
150336,ZRGYO.IS,2026-07-10,2026-07-13 11:30:42.704304+00:00,17.7200,18.3200,17.6700,18.0500,4061994.0000,18.0500,17.5300,17.8900,17.5000,1072686.0000,0,-0.5000,-0.8660,0.8660,0.5000,0.8978,-0.4404,-0.7908,-0.6121,-0.1369,-0.9906,0.2712,0.9625,53.3132,17.7220,18.3198,4335958.1500,18.4700,16.5000,16.9686,False,False,False,False,False,False,False,False,True,False,True,False,False,None,17.7200,17.8900,17.5000,0.0000,NaN,0.6500,0.5764
150337,ZRGYO.IS,2026-07-13,2026-07-13 11:30:42.704304+00:00,18.0500,18.0500,17.5200,17.6400,681750.0000,17.6400,17.7200,18.3200,17.6700,4061994.0000,2,-0.5000,-0.8660,0.8660,0.5000,0.4853,-0.8743,-0.8486,0.5290,-0.1877,-0.9822,0.3688,0.9295,47.6666,17.7315,18.2350,4042407.5500,18.4700,16.7200,16.1758,False,False,False,False,False,False,False,False,True,False,True,False,False,None,18.0500,18.3200,17.6700,0.0000,NaN,0.5300,0.5643


## 5. Common Gap Fill Validation

In [260]:
def build_common_gap_fill_events(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for symbol, symbol_frame in frame.sort_values(["symbol", "date"]).groupby("symbol", observed=True):
        symbol_frame = symbol_frame.reset_index(drop=True)
        common_events = symbol_frame[symbol_frame["Gap_Type"].isin(["Common Up", "Common Down"])]
        for event_index, event in common_events.iterrows():
            target = event["common_gap_fill_target"]
            future = symbol_frame.iloc[event_index + 1 :]
            if pd.isna(target) or future.empty:
                hit_index = None
            elif event["Gap_Type"] == "Common Up":
                hits = future[future["low"] <= target]
                hit_index = None if hits.empty else int(hits.index[0])
            else:
                hits = future[future["high"] >= target]
                hit_index = None if hits.empty else int(hits.index[0])

            rows.append(
                {
                    "symbol": symbol,
                    "Gap_Type": event["Gap_Type"],
                    "entry_time": event["date"],
                    "entry_open": event["open"],
                    "target": target,
                    "filled": hit_index is not None,
                    "fill_time": None if hit_index is None else symbol_frame.loc[hit_index, "date"],
                    "bars_to_fill": None if hit_index is None else hit_index - event_index,
                    "abs_gap_pct": abs(event["gap_pct"]),
                    "target_distance_pct": abs(event["open"] - target) / event["open"] if pd.notna(target) else np.nan,
                }
            )
    return pd.DataFrame(rows)

common_fill_events = build_common_gap_fill_events(features)
common_fill_report = (
    common_fill_events
    .groupby(["symbol", "Gap_Type"], observed=True)
    .agg(
        events=("entry_time", "size"),
        fill_rate=("filled", "mean"),
        median_bars_to_fill=("bars_to_fill", "median"),
        median_abs_gap_pct=("abs_gap_pct", "median"),
        avg_abs_gap_pct=("abs_gap_pct", "mean"),
        avg_target_distance_pct=("target_distance_pct", "mean"),
    )
    .reset_index()
)

display(common_fill_report)
log_step(
    "common_gap_fill_validation",
    total_common_events=len(common_fill_events),
    filled_events=int(common_fill_events["filled"].sum()) if not common_fill_events.empty else 0,
    median_bars_to_fill=common_fill_events["bars_to_fill"].median() if "bars_to_fill" in common_fill_events.columns else np.nan,
)
px.bar(
    common_fill_report,
    x="Gap_Type",
    y="fill_rate",
    color="symbol",
    barmode="group",
    text_auto=".1%",
    title="Forward Common Gap Fill Rate By 5-Minute Bars",
).show()


,symbol,Gap_Type,events,fill_rate,median_bars_to_fill,median_abs_gap_pct,avg_abs_gap_pct,avg_target_distance_pct
0,A1CAP.IS,Common Down,5,0.8000,9.0000,0.0149,0.0307,0.0057
1,A1CAP.IS,Common Up,3,1.0000,3.0000,0.0234,0.0269,0.0177
2,A1YEN.IS,Common Down,3,1.0000,2.0000,0.0108,0.0116,0.0107
3,A1YEN.IS,Common Up,2,1.0000,5.5000,0.0172,0.0172,0.0115
4,AAGYO.IS,Common Down,2,0.5000,2.0000,0.0067,0.0067,0.0062
...,...,...,...,...,...,...,...,...
989,ZGYO.IS,Common Down,1,1.0000,11.0000,0.0406,0.0406,0.0423
990,ZGYO.IS,Common Up,9,0.4444,11.5000,0.0996,0.0881,0.0806
991,ZOREN.IS,Common Down,2,1.0000,2.0000,0.0168,0.0168,0.0116
992,ZOREN.IS,Common Up,2,1.0000,1.0000,0.0171,0.0171,0.0067



[common_gap_fill_validation]
  total_common_events: 3974
  filled_events: 3480
  median_bars_to_fill: 3.0


## 6. Gap Signal Candlesticks

In [261]:
plot_window = 500
marker_styles = {
    "Breakaway Up": dict(symbol="triangle-up", color="#16a34a", size=12),
    "Runaway Up": dict(symbol="triangle-up", color="#22c55e", size=10),
    "Common Down": dict(symbol="circle", color="#0284c7", size=8),
    "Common Up": dict(symbol="circle", color="#f97316", size=8),
    "Exhaustion Up": dict(symbol="x", color="#dc2626", size=12),
    "Breakaway Down": dict(symbol="triangle-down", color="#7f1d1d", size=11),
    "Runaway Down": dict(symbol="triangle-down", color="#991b1b", size=10),
}

log_step("signal_plot_selection", plot_window=plot_window, plotted_symbols=PLOT_SYMBOLS)

for symbol in PLOT_SYMBOLS:
    sample = features[features["symbol"] == symbol].tail(plot_window).copy()
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.72, 0.28], vertical_spacing=0.03)
    fig.add_trace(
        go.Candlestick(
            x=sample["date"],
            open=sample["open"],
            high=sample["high"],
            low=sample["low"],
            close=sample["close"],
            name=symbol,
        ),
        row=1,
        col=1,
    )
    for gap_type, style in marker_styles.items():
        points = sample[sample["Gap_Type"] == gap_type]
        if points.empty:
            continue
        fig.add_trace(
            go.Scatter(
                x=points["date"],
                y=points["high"] * 1.012,
                mode="markers",
                marker=style,
                name=gap_type,
                text=points["Gap_Type"],
            ),
            row=1,
            col=1,
        )
    fig.add_trace(go.Bar(x=sample["date"], y=sample["volume"], name="Volume", marker_color="#64748b"), row=2, col=1)
    fig.update_layout(title=f"{symbol} Conventional Gap Signals", xaxis_rangeslider_visible=False, height=760)
    fig.show()



[signal_plot_selection]
  plot_window: 500
  plotted_symbols: ['TURSG.IS', 'HLGYO.IS', 'ENTRA.IS', 'OSTIM.IS', 'GOODY.IS']


## 7. Research Signals And Portfolio Weights

Exit yaklaşımı:
- Common gap: gap kapanma hipotezi nedeniyle kısa holding. Conservative vectorbt uygulamasında ertesi bar çıkış sinyali kullanılır; same-day fill ayrıca yukarıda validasyonlandı.
- Breakaway/Runaway Up: trend long. SMA short altına kapanış, RSI aşırı ısınma veya Exhaustion Up gelirse çıkış.
- Exhaustion Up short: sadece teyitli ise açılır; close > SMA_Short veya RSI normalize olursa çıkış. Daha düşük güvenli olduğu için sizing zaten risk-budget ile sınırlı.

In [262]:
all_dates = pd.DatetimeIndex(sorted(pd.to_datetime(features["date"].dropna().unique())), name="date")

close = features.pivot(index="date", columns="symbol", values="close").reindex(all_dates).ffill()
open_ = features.pivot(index="date", columns="symbol", values="open").reindex(all_dates).ffill()
high = features.pivot(index="date", columns="symbol", values="high").reindex(all_dates).ffill()
low = features.pivot(index="date", columns="symbol", values="low").reindex(all_dates).ffill()
volume = features.pivot(index="date", columns="symbol", values="volume").reindex(all_dates).fillna(0)
atr = features.pivot(index="date", columns="symbol", values="ATR").reindex(all_dates).ffill().bfill()
rsi = features.pivot(index="date", columns="symbol", values="RSI").reindex(all_dates).ffill()
sma_short = features.pivot(index="date", columns="symbol", values="SMA_Short").reindex(all_dates).ffill()
sma_long = features.pivot(index="date", columns="symbol", values="SMA_Long").reindex(all_dates).ffill()
adx = features.pivot(index="date", columns="symbol", values="ADX").reindex(all_dates).ffill()
gap_pct = features.pivot(index="date", columns="symbol", values="gap_pct").reindex(all_dates).fillna(0.0)
gap_type = features.pivot(index="date", columns="symbol", values="Gap_Type").reindex(all_dates).fillna("None")
prev_high_target = features.pivot(index="date", columns="symbol", values="prev_high_research").reindex(all_dates).ffill()
prev_low_target = features.pivot(index="date", columns="symbol", values="prev_low_research").reindex(all_dates).ffill()
common_fill_target = features.pivot(index="date", columns="symbol", values="common_gap_fill_target").reindex(all_dates)

common_down = gap_type.eq("Common Down") & gap_pct.abs().ge(MIN_GAP_ABS)
common_up = gap_type.eq("Common Up") & gap_pct.abs().ge(MIN_GAP_ABS)


def build_target_exit_state(
    entry_targets: pd.DataFrame,
    hit_prices: pd.DataFrame,
    hit_direction: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    exits = pd.DataFrame(False, index=entry_targets.index, columns=entry_targets.columns)
    active_targets = pd.DataFrame(np.nan, index=entry_targets.index, columns=entry_targets.columns)

    for symbol in entry_targets.columns:
        active_target = np.nan
        for timestamp in entry_targets.index:
            if pd.notna(active_target):
                active_targets.at[timestamp, symbol] = active_target
                price = hit_prices.at[timestamp, symbol]
                target_hit = price <= active_target if hit_direction == "down" else price >= active_target
                if pd.notna(price) and target_hit:
                    exits.at[timestamp, symbol] = True
                    active_target = np.nan
                    continue

            new_target = entry_targets.at[timestamp, symbol]
            if pd.notna(new_target) and pd.isna(active_target):
                active_target = new_target

    return exits, active_targets

common_up_entry_target = prev_high_target.where(common_up)
common_down_entry_target = prev_low_target.where(common_down)
common_up_target_hit, active_common_up_target = build_target_exit_state(
    common_up_entry_target,
    low,
    hit_direction="down",
)
common_down_target_hit, active_common_down_target = build_target_exit_state(
    common_down_entry_target,
    high,
    hit_direction="up",
)
common_target_hit_price = active_common_up_target.where(common_up_target_hit).combine_first(
    active_common_down_target.where(common_down_target_hit)
)

trend_up = gap_type.isin(["Breakaway Up", "Runaway Up"]) & adx.ge(20) & rsi.lt(76)
trend_down = gap_type.isin(["Breakaway Down", "Runaway Down"]) & adx.ge(20) & rsi.gt(24)
exhaustion_up_short = gap_type.eq("Exhaustion Up") & rsi.gt(65) & close.lt(open_)

common_down_fill_long = common_down
common_up_fill_short = common_up
common_long_entries = common_down_fill_long
common_short_entries = common_up_fill_short
trend_long_entries = trend_up
trend_short_entries = trend_down & close.lt(sma_short)
exhaustion_short_entries = exhaustion_up_short

long_entries = trend_long_entries | common_long_entries
short_entries = trend_short_entries | common_short_entries | exhaustion_short_entries

common_fill_long_exits = common_down_target_hit
common_fill_short_exits = common_up_target_hit
trend_long_exits = close.lt(sma_short) | rsi.gt(78) | gap_type.eq("Exhaustion Up")
trend_short_exits = close.gt(sma_short) | rsi.lt(35) | gap_type.eq("Exhaustion Down")
exhaustion_short_exits = close.gt(sma_short) | rsi.lt(50)

common_long_exits = common_fill_long_exits
common_short_exits = common_fill_short_exits
long_exits = common_fill_long_exits | trend_long_exits
short_exits = common_fill_short_exits | trend_short_exits | exhaustion_short_exits

signal_report = pd.DataFrame(
    {
        "long_trend_breakaway_runaway": trend_long_entries.sum(),
        "long_common_down_fill": common_down_fill_long.sum(),
        "short_common_up_fill": common_up_fill_short.sum(),
        "short_trend_down": trend_short_entries.sum(),
        "short_exhaustion_up": exhaustion_short_entries.sum(),
        "common_up_target_hits_after_entry": common_up_target_hit.sum(),
        "common_down_target_hits_after_entry": common_down_target_hit.sum(),
    }
).T
signal_report.columns.name = "symbol"
display(signal_report)
log_step(
    "signal_matrix_ready",
    timeline_rows=len(all_dates),
    symbol_count=len(close.columns),
    long_entry_events=int(long_entries.sum().sum()),
    short_entry_events=int(short_entries.sum().sum()),
    long_exit_events=int(long_exits.sum().sum()),
    short_exit_events=int(short_exits.sum().sum()),
)


symbol,A1CAP.IS,A1YEN.IS,AAGYO.IS,ACSEL.IS,ADEL.IS,ADESE.IS,ADGYO.IS,AEFES.IS,AFYON.IS,AGESA.IS,AGHOL.IS,AGROT.IS,AGYO.IS,AHGAZ.IS,AHSGY.IS,AKBNK.IS,AKCNS.IS,AKENR.IS,AKFGY.IS,AKFIS.IS,AKFYE.IS,AKGRT.IS,AKHAN.IS,AKMGY.IS,AKSA.IS,AKSEN.IS,AKSGY.IS,AKSUE.IS,AKYHO.IS,ALARK.IS,ALBRK.IS,ALCAR.IS,ALCTL.IS,ALFAS.IS,ALGYO.IS,ALKA.IS,ALKIM.IS,ALKLC.IS,ALTNY.IS,ALVES.IS,ANELE.IS,ANGEN.IS,ANHYT.IS,ANSGR.IS,ARASE.IS,ARCLK.IS,ARDYZ.IS,ARENA.IS,ARFYE.IS,ARMGD.IS,ARSAN.IS,ARTMS.IS,ARZUM.IS,ASELS.IS,ASGYO.IS,ASTOR.IS,ASUZU.IS,ATAGY.IS,ATAKP.IS,ATATP.IS,...,TRALT.IS,TRCAS.IS,TRENJ.IS,TRGYO.IS,TRHOL.IS,TRILC.IS,TRMET.IS,TSGYO.IS,TSKB.IS,TSPOR.IS,TTKOM.IS,TTRAK.IS,TUCLK.IS,TUKAS.IS,TUPRS.IS,TUREX.IS,TURGG.IS,TURSG.IS,UCAYM.IS,UFUK.IS,ULAS.IS,ULKER.IS,ULUFA.IS,ULUSE.IS,ULUUN.IS,UNLU.IS,USAK.IS,VAKBN.IS,VAKFA.IS,VAKFN.IS,VAKKO.IS,VANGD.IS,VBTYZ.IS,VERUS.IS,VESBE.IS,VESTL.IS,VKFYO.IS,VKGYO.IS,VKING.IS,VRGYO.IS,VSNMD.IS,YAPRK.IS,YATAS.IS,YAYLA.IS,YBTAS.IS,YEOTK.IS,YESIL.IS,YGGYO.IS,YIGIT.IS,YKBNK.IS,YKSLN.IS,YONGA.IS,YUNSA.IS,YYAPI.IS,YYLGD.IS,ZEDUR.IS,ZERGY.IS,ZGYO.IS,ZOREN.IS,ZRGYO.IS
long_trend_breakaway_runaway,2,2,0,1,4,0,0,1,1,0,1,0,0,0,0,3,0,1,0,0,3,2,0,0,1,0,1,1,0,0,1,0,1,1,1,1,0,0,2,1,1,0,0,0,1,0,2,0,0,1,1,0,0,2,1,2,0,0,0,0,...,3,0,2,2,2,0,2,0,0,0,1,0,0,1,2,1,1,0,1,0,1,0,1,3,0,0,0,1,0,2,0,1,1,1,1,0,3,0,0,0,2,2,0,0,1,0,0,1,0,3,1,6,1,0,1,0,2,1,1,1
long_common_down_fill,5,3,1,1,3,5,0,1,4,1,3,2,1,0,0,2,1,1,1,3,2,1,1,1,0,1,1,0,1,2,3,0,1,2,3,3,1,4,4,0,3,3,2,3,3,4,2,1,1,2,3,0,1,0,1,2,2,0,1,2,...,6,1,5,3,5,7,5,2,2,0,3,1,2,0,2,0,2,2,3,1,1,4,2,2,0,0,2,4,1,2,2,1,1,4,1,2,2,4,1,1,6,2,1,2,13,2,1,1,1,2,1,13,0,3,2,1,2,1,2,1
short_common_up_fill,3,2,2,3,1,2,1,2,3,2,2,2,0,0,3,2,2,0,3,0,3,2,3,0,2,3,1,0,0,6,1,4,2,1,0,1,0,6,1,4,5,2,4,0,1,5,2,3,6,0,2,2,2,0,5,2,2,0,2,6,...,4,0,4,3,9,5,3,3,7,0,4,2,2,2,2,2,2,1,7,2,0,4,1,3,0,1,2,6,4,1,1,1,0,3,3,6,4,2,5,3,2,3,0,4,19,7,5,0,2,3,3,24,1,6,1,1,4,9,2,0
short_trend_down,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,2,0,0,1,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,13,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1
short_exhaustion_up,1,1,0,1,1,0,0,0,0,0,1,0,0,0,3,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,1,0,0,0,1,1,0,0,1,0,1,0,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,1,0,2,1,0,0,0,0
common_up_target_hits_after_entry,3,2,2,2,1,2,0,2,3,1,2,2,0,0,3,2,1,0,3,0,1,2,1,0,2,1,0,0,0,6,1,4,1,1,0,1,0,2,1,4,1,2,3,0,0,5,0,2,0,0,1,2,2,0,5,2,2,0,2,0,...,3,0,4,0,1,4,0,3,6,0,4,2,2,2,2,2,2,1,0,0,0,4,1,3,0,1,2,2,3,1,0,1,0,0,3,6,4,2,3,2,1,2,0,3,11,2,2,0,2,3,3,14,1,6,1,1,4,0,2,0
common_down_target_hits_after_entry,2,3,1,1,3,1,0,1,1,1,3,2,1,0,0,1,1,1,1,2,2,1,1,0,0,1,1,0,1,2,2,0,1,2,3,2,0,2,0,0,3,3,2,2,3,4,2,1,1,2,3,0,1,0,1,2,1,0,1,2,...,6,0,4,3,5,1,5,2,2,0,3,1,0,0,2,0,1,2,3,1,1,2,1,2,0,0,2,1,1,2,2,1,1,2,1,1,1,2,1,1,0,2,1,2,3,2,1,1,1,1,1,10,0,2,2,1,1,1,2,1



[signal_matrix_ready]
  timeline_rows: 257
  symbol_count: 610
  long_entry_events: 2257
  short_entry_events: 2442
  long_exit_events: 78130
  short_exit_events: 133233


In [263]:
returns = close.pct_change().dropna(how="all").fillna(0.0)
train_end = returns.index[int(len(returns) * TRAIN_FRACTION)]
train_returns = returns.loc[:train_end]

mu = train_returns.mean().values
cov = train_returns.cov().values
n_assets = len(close.columns)
observed_bars_per_day = returns.groupby(returns.index.date).size().median() if len(returns) else np.nan
periods_per_year = 252 * observed_bars_per_day if pd.notna(observed_bars_per_day) and observed_bars_per_day > 0 else 252

rng = np.random.default_rng(42)
frontier_samples = rng.dirichlet(np.ones(n_assets), size=EFFICIENT_FRONTIER_TRIALS)
frontier = pd.DataFrame(
    {
        "trial": np.arange(EFFICIENT_FRONTIER_TRIALS),
        "bar_return": frontier_samples @ mu,
        "bar_volatility": np.sqrt(np.einsum("ij,jk,ik->i", frontier_samples, cov, frontier_samples)),
    }
)
frontier["annualized_return"] = frontier["bar_return"] * periods_per_year
frontier["annualized_volatility"] = frontier["bar_volatility"] * np.sqrt(periods_per_year)
frontier["sharpe"] = np.divide(
    frontier["bar_return"] - RISK_FREE_DAILY,
    frontier["bar_volatility"],
    out=np.full(len(frontier), np.nan),
    where=frontier["bar_volatility"].to_numpy() > 0,
)
for asset_index, symbol in enumerate(close.columns):
    frontier[f"w_{symbol}"] = frontier_samples[:, asset_index]

if minimize is not None and n_assets > 1:
    def negative_sharpe(weights: np.ndarray) -> float:
        port_return = float(weights @ mu)
        port_vol = float(np.sqrt(weights @ cov @ weights))
        if port_vol <= 0:
            return 1e9
        return -((port_return - RISK_FREE_DAILY) / port_vol)

    result = minimize(
        negative_sharpe,
        x0=np.repeat(1 / n_assets, n_assets),
        method="SLSQP",
        bounds=[(0.0, 1.0)] * n_assets,
        constraints={"type": "eq", "fun": lambda weights: float(weights.sum() - 1.0)},
    )
    optimized_weights = result.x if result.success else frontier_samples[int(frontier["sharpe"].idxmax())]
else:
    optimized_weights = frontier_samples[int(frontier["sharpe"].idxmax())]

weights = pd.Series(optimized_weights, index=close.columns, name="optimized_weight")
weights = weights / weights.sum()
best_frontier_row = frontier.loc[frontier["sharpe"].idxmax()]
log_step(
    "efficient_frontier_result",
    trials=len(frontier),
    train_rows=len(train_returns),
    train_start=train_returns.index.min(),
    train_end=train_returns.index.max(),
    best_trial=int(best_frontier_row["trial"]),
    best_trial_sharpe=round(float(best_frontier_row["sharpe"]), 4),
    optimized_weight_top=weights.sort_values(ascending=False).head(10).round(4).to_dict(),
)
display(weights.to_frame())
display(frontier.nlargest(10, "sharpe"))

hover_weight_symbols = [symbol for symbol in PLOT_SYMBOLS if f"w_{symbol}" in frontier.columns]
hover_data = {f"w_{symbol}": ":.1%" for symbol in hover_weight_symbols}
hover_data.update({"trial": True, "annualized_return": ":.2%", "annualized_volatility": ":.2%", "sharpe": ":.3f"})
fig = px.scatter(
    frontier,
    x="annualized_volatility",
    y="annualized_return",
    color="sharpe",
    hover_data=hover_data,
    title="Efficient Frontier Trials - Train Window",
    labels={
        "annualized_volatility": "Annualized Volatility",
        "annualized_return": "Annualized Return",
    },
)
opt_return = float(weights.values @ mu * periods_per_year)
opt_vol = float(np.sqrt(weights.values @ cov @ weights.values) * np.sqrt(periods_per_year))
equal_weights = np.repeat(1 / n_assets, n_assets)
equal_return = float(equal_weights @ mu * periods_per_year)
equal_vol = float(np.sqrt(equal_weights @ cov @ equal_weights) * np.sqrt(periods_per_year))
fig.add_trace(go.Scatter(x=[opt_vol], y=[opt_return], mode="markers", marker=dict(size=17, color="red", symbol="star"), name="Optimized"))
fig.add_trace(go.Scatter(x=[equal_vol], y=[equal_return], mode="markers", marker=dict(size=13, color="black", symbol="x"), name="Equal Weight"))
fig.show()


/tmp/ipykernel_6286/4016829690.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  frontier[f"w_{symbol}"] = frontier_samples[:, asset_index]
/tmp/ipykernel_6286/4016829690.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  frontier[f"w_{symbol}"] = frontier_samples[:, asset_index]
/tmp/ipykernel_6286/4016829690.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) in


[efficient_frontier_result]
  trials: 20000
  train_rows: 180
  train_start: 2025-07-16 00:00:00
  train_end: 2026-03-27 00:00:00
  best_trial: 17703
  best_trial_sharpe: 0.1668
  optimized_weight_top: {'ODINE.IS': 0.0808, 'ECOGR.IS': 0.0678, 'GUNDG.IS': 0.0539, 'ZGYO.IS': 0.0405, 'IEYHO.IS': 0.038, 'BARMA.IS': 0.0369, 'EKIZ.IS': 0.0359, 'IZFAS.IS': 0.0324, 'BSOKE.IS': 0.031, 'CEMZY.IS': 0.0305}


,optimized_weight
symbol,
A1CAP.IS,0.0000
A1YEN.IS,0.0000
AAGYO.IS,0.0000
ACSEL.IS,0.0000
ADEL.IS,0.0000
...,...
ZEDUR.IS,0.0000
ZERGY.IS,0.0000
ZGYO.IS,0.0405


,trial,bar_return,bar_volatility,annualized_return,annualized_volatility,sharpe,w_A1CAP.IS,w_A1YEN.IS,w_AAGYO.IS,w_ACSEL.IS,w_ADEL.IS,w_ADESE.IS,w_ADGYO.IS,w_AEFES.IS,w_AFYON.IS,w_AGESA.IS,w_AGHOL.IS,w_AGROT.IS,w_AGYO.IS,w_AHGAZ.IS,w_AHSGY.IS,w_AKBNK.IS,w_AKCNS.IS,w_AKENR.IS,w_AKFGY.IS,w_AKFIS.IS,w_AKFYE.IS,w_AKGRT.IS,w_AKHAN.IS,w_AKMGY.IS,w_AKSA.IS,w_AKSEN.IS,w_AKSGY.IS,w_AKSUE.IS,w_AKYHO.IS,w_ALARK.IS,w_ALBRK.IS,w_ALCAR.IS,w_ALCTL.IS,w_ALFAS.IS,w_ALGYO.IS,w_ALKA.IS,w_ALKIM.IS,w_ALKLC.IS,w_ALTNY.IS,w_ALVES.IS,w_ANELE.IS,w_ANGEN.IS,w_ANHYT.IS,w_ANSGR.IS,w_ARASE.IS,w_ARCLK.IS,w_ARDYZ.IS,w_ARENA.IS,w_ARFYE.IS,w_ARMGD.IS,w_ARSAN.IS,w_ARTMS.IS,w_ARZUM.IS,w_ASELS.IS,...,w_TRALT.IS,w_TRCAS.IS,w_TRENJ.IS,w_TRGYO.IS,w_TRHOL.IS,w_TRILC.IS,w_TRMET.IS,w_TSGYO.IS,w_TSKB.IS,w_TSPOR.IS,w_TTKOM.IS,w_TTRAK.IS,w_TUCLK.IS,w_TUKAS.IS,w_TUPRS.IS,w_TUREX.IS,w_TURGG.IS,w_TURSG.IS,w_UCAYM.IS,w_UFUK.IS,w_ULAS.IS,w_ULKER.IS,w_ULUFA.IS,w_ULUSE.IS,w_ULUUN.IS,w_UNLU.IS,w_USAK.IS,w_VAKBN.IS,w_VAKFA.IS,w_VAKFN.IS,w_VAKKO.IS,w_VANGD.IS,w_VBTYZ.IS,w_VERUS.IS,w_VESBE.IS,w_VESTL.IS,w_VKFYO.IS,w_VKGYO.IS,w_VKING.IS,w_VRGYO.IS,w_VSNMD.IS,w_YAPRK.IS,w_YATAS.IS,w_YAYLA.IS,w_YBTAS.IS,w_YEOTK.IS,w_YESIL.IS,w_YGGYO.IS,w_YIGIT.IS,w_YKBNK.IS,w_YKSLN.IS,w_YONGA.IS,w_YUNSA.IS,w_YYAPI.IS,w_YYLGD.IS,w_ZEDUR.IS,w_ZERGY.IS,w_ZGYO.IS,w_ZOREN.IS,w_ZRGYO.IS
17703,17703,0.0018,0.0109,0.4597,0.1736,0.1668,0.0026,0.0000,0.0009,0.0024,0.0018,0.0009,0.0007,0.0010,0.0002,0.0035,0.0010,0.0015,0.0009,0.0013,0.0025,0.0006,0.0018,0.0002,0.0005,0.0021,0.0010,0.0019,0.0002,0.0012,0.0036,0.0007,0.0011,0.0001,0.0012,0.0009,0.0053,0.0024,0.0013,0.0001,0.0017,0.0015,0.0013,0.0016,0.0007,0.0022,0.0020,0.0003,0.0035,0.0050,0.0022,0.0001,0.0017,0.0001,0.0008,0.0017,0.0003,0.0000,0.0007,0.0001,...,0.0005,0.0004,0.0027,0.0003,0.0021,0.0020,0.0013,0.0007,0.0016,0.0005,0.0004,0.0009,0.0023,0.0001,0.0015,0.0008,0.0001,0.0013,0.0069,0.0033,0.0008,0.0001,0.0015,0.0015,0.0008,0.0012,0.0012,0.0014,0.0025,0.0015,0.0026,0.0002,0.0021,0.0015,0.0038,0.0004,0.0016,0.0008,0.0011,0.0003,0.0001,0.0014,0.0063,0.0043,0.0006,0.0006,0.0002,0.0026,0.0003,0.0012,0.0004,0.0012,0.0025,0.0013,0.0005,0.0095,0.0038,0.0017,0.0001,0.0031
2402,2402,0.0019,0.0113,0.4734,0.1788,0.1668,0.0004,0.0002,0.0025,0.0032,0.0002,0.0034,0.0004,0.0001,0.0001,0.0002,0.0033,0.0008,0.0012,0.0040,0.0002,0.0000,0.0058,0.0020,0.0005,0.0049,0.0003,0.0003,0.0010,0.0008,0.0017,0.0003,0.0009,0.0001,0.0029,0.0002,0.0001,0.0027,0.0025,0.0036,0.0012,0.0014,0.0044,0.0021,0.0029,0.0003,0.0001,0.0007,0.0003,0.0011,0.0003,0.0079,0.0013,0.0001,0.0008,0.0021,0.0037,0.0016,0.0022,0.0003,...,0.0027,0.0015,0.0035,0.0018,0.0019,0.0002,0.0010,0.0011,0.0005,0.0000,0.0000,0.0015,0.0001,0.0021,0.0022,0.0002,0.0012,0.0037,0.0000,0.0001,0.0046,0.0023,0.0009,0.0019,0.0007,0.0005,0.0007,0.0009,0.0009,0.0013,0.0046,0.0002,0.0017,0.0016,0.0014,0.0008,0.0035,0.0015,0.0005,0.0003,0.0004,0.0001,0.0006,0.0007,0.0001,0.0019,0.0029,0.0003,0.0006,0.0003,0.0027,0.0019,0.0003,0.0022,0.0003,0.0000,0.0044,0.0046,0.0013,0.0010
6981,6981,0.0018,0.0109,0.4517,0.1723,0.1651,0.0023,0.0016,0.0034,0.0004,0.0001,0.0028,0.0011,0.0011,0.0009,0.0020,0.0004,0.0000,0.0008,0.0014,0.0007,0.0026,0.0081,0.0007,0.0001,0.0010,0.0008,0.0038,0.0017,0.0039,0.0000,0.0026,0.0019,0.0002,0.0002,0.0014,0.0011,0.0012,0.0004,0.0008,0.0036,0.0004,0.0036,0.0010,0.0001,0.0013,0.0006,0.0010,0.0003,0.0017,0.0021,0.0016,0.0002,0.0029,0.0001,0.0002,0.0011,0.0002,0.0002,0.0008,...,0.0020,0.0000,0.0037,0.0032,0.0069,0.0003,0.0023,0.0009,0.0039,0.0022,0.0033,0.0021,0.0026,0.0025,0.0011,0.0011,0.0005,0.0027,0.0000,0.0015,0.0053,0.0021,0.0009,0.0038,0.0027,0.0003,0.0047,0.0011,0.0001,0.0024,0.0020,0.0023,0.0021,0.0018,0.0010,0.0019,0.0018,0.0058,0.0003,0.0046,0.0010,0.0002,0.0032,0.0021,0.0006,0.0004,0.0011,0.0011,0.0004,0.0008,0.0018,0.0017,0.0007,0.0006,0.0008,0.0017,0.0011,0.0037,0.0016,0.0047
14750,14750,0.0018,0.0112,0.4570,0.1781,0.1616,0.0007,0.0016,0.0023,0.0095,0.0008,0.0008,0.0022,0.0055,0.0001,0.0032,0.0007,0.001

## 8. Dynamic Risk Sizing

In [264]:
stop_distance = (atr * STOP_ATR_MULTIPLIER).clip(lower=close * 0.005)
stop_pct = (stop_distance / close).clip(lower=0.005, upper=0.15)
tp_pct = (stop_pct * TAKE_PROFIT_R_MULTIPLIER).clip(upper=0.35)

integer_leverage = np.ceil(gap_pct.abs() / GAP_LEVERAGE_BUCKET).clip(1, MAX_INTEGER_LEVERAGE).astype(int)
risk_budget_by_symbol = pd.Series(MAX_POSITION_RISK_TL, index=close.columns) * weights

shares_by_risk = pd.DataFrame(index=close.index, columns=close.columns, dtype=float)
shares_by_cash = pd.DataFrame(index=close.index, columns=close.columns, dtype=float)
for symbol in close.columns:
    shares_by_risk[symbol] = np.floor(risk_budget_by_symbol[symbol] / stop_distance[symbol].replace(0, np.nan))
    shares_by_cash[symbol] = np.floor((INITIAL_CAPITAL * weights[symbol] * integer_leverage[symbol]) / open_[symbol].replace(0, np.nan))

position_size = np.minimum(shares_by_risk, shares_by_cash).replace([np.inf, -np.inf], np.nan).fillna(0.0)
position_size = position_size.where(long_entries | short_entries, 0.0)

risk_check = (position_size * stop_distance).where(long_entries | short_entries, 0.0)
risk_summary = pd.DataFrame({
    "max_single_signal_risk_tl": risk_check.max(),
    "risk_budget_tl": risk_budget_by_symbol,
    "max_integer_leverage_seen": integer_leverage.max(),
})
display(risk_summary)
log_step(
    "dynamic_risk_sizing",
    max_position_size=float(position_size.max().max()),
    total_signal_count=int((long_entries | short_entries).sum().sum()),
    max_single_signal_risk_tl=round(float(risk_check.max().max()), 2),
    max_integer_leverage_seen=int(integer_leverage.max().max()),
)
print(f"Total risk budget across simultaneous positions is capped by weights at <= {MAX_POSITION_RISK_TL:,.0f} TL.")


,max_single_signal_risk_tl,risk_budget_tl,max_integer_leverage_seen
symbol,,,
A1CAP.IS,0.0000,0.0000,2
A1YEN.IS,0.0000,0.0000,2
AAGYO.IS,0.0000,0.0374,2
ACSEL.IS,0.0000,0.0000,2
ADEL.IS,0.0000,0.0000,2
...,...,...,...
ZEDUR.IS,0.0000,0.0000,2
ZERGY.IS,0.0000,0.0000,2
ZGYO.IS,40.3200,40.4633,2



[dynamic_risk_sizing]
  max_position_size: 106.0
  total_signal_count: 4699
  max_single_signal_risk_tl: 75.75
  max_integer_leverage_seen: 3
Total risk budget across simultaneous positions is capped by weights at <= 1,000 TL.


## 9. Vectorbt Backtests

In [265]:
def bool_frame(frame: pd.DataFrame) -> pd.DataFrame:
    return (
        pd.DataFrame(frame, index=close.index, columns=close.columns)
        .fillna(False)
        .astype(bool)
    )


def float_frame(frame: pd.DataFrame, fill_value: float = 0.0) -> pd.DataFrame:
    return (
        pd.DataFrame(frame, index=close.index, columns=close.columns)
        .replace([np.inf, -np.inf], np.nan)
        .fillna(fill_value)
        .astype(np.float64)
    )


close = float_frame(close, fill_value=np.nan).ffill().bfill()
open_ = float_frame(open_, fill_value=np.nan).ffill().bfill()
high = float_frame(high, fill_value=np.nan).ffill().bfill()
low = float_frame(low, fill_value=np.nan).ffill().bfill()
sma_short = float_frame(sma_short, fill_value=np.nan).ffill().bfill()
sma_long = float_frame(sma_long, fill_value=np.nan).ffill().bfill()
position_size = float_frame(position_size, fill_value=0.0)
stop_pct = float_frame(stop_pct, fill_value=0.0).clip(lower=0.0)
tp_pct = float_frame(tp_pct, fill_value=0.0).clip(lower=0.0)

long_entries = bool_frame(long_entries)
long_exits = bool_frame(long_exits)
short_entries = bool_frame(short_entries)
short_exits = bool_frame(short_exits)
trend_long_entries = bool_frame(trend_long_entries)
trend_long_exits = bool_frame(trend_long_exits)
common_long_entries = bool_frame(common_long_entries)
common_long_exits = bool_frame(common_long_exits)
common_short_entries = bool_frame(common_short_entries)
common_short_exits = bool_frame(common_short_exits)
exhaustion_short_entries = bool_frame(exhaustion_short_entries)
exhaustion_short_exits = bool_frame(exhaustion_short_exits)

pf_gap = vbt.Portfolio.from_signals(
    close=close,
    entries=long_entries,
    exits=long_exits,
    short_entries=short_entries,
    short_exits=short_exits,
    size=position_size,
    size_type="amount",
    init_cash=INITIAL_CAPITAL,
    fees=FEES,
    slippage=SLIPPAGE,
    sl_stop=stop_pct,
    tp_stop=tp_pct,
    freq=VECTORBT_FREQ,
    group_by=True,
    cash_sharing=True,
    call_seq="auto",
)

pf_long_trend = vbt.Portfolio.from_signals(
    close=close,
    entries=trend_long_entries,
    exits=trend_long_exits,
    size=float_frame(position_size.where(trend_long_entries, 0.0), 0.0),
    size_type="amount",
    init_cash=INITIAL_CAPITAL,
    fees=FEES,
    slippage=SLIPPAGE,
    sl_stop=stop_pct,
    tp_stop=tp_pct,
    freq=VECTORBT_FREQ,
    group_by=True,
    cash_sharing=True,
    call_seq="auto",
)

pf_common = vbt.Portfolio.from_signals(
    close=close,
    entries=common_long_entries,
    exits=common_long_exits,
    short_entries=common_short_entries,
    short_exits=common_short_exits,
    size=float_frame(position_size.where(common_long_entries | common_short_entries, 0.0), 0.0),
    size_type="amount",
    init_cash=INITIAL_CAPITAL,
    fees=FEES,
    slippage=SLIPPAGE,
    freq=VECTORBT_FREQ,
    group_by=True,
    cash_sharing=True,
    call_seq="auto",
)

pf_exhaustion_short = vbt.Portfolio.from_signals(
    close=close,
    short_entries=exhaustion_short_entries,
    short_exits=exhaustion_short_exits,
    size=float_frame(position_size.where(exhaustion_short_entries, 0.0), 0.0),
    size_type="amount",
    init_cash=INITIAL_CAPITAL,
    fees=FEES,
    slippage=SLIPPAGE,
    sl_stop=stop_pct,
    tp_stop=tp_pct,
    freq=VECTORBT_FREQ,
    group_by=True,
    cash_sharing=True,
    call_seq="auto",
)

sma_entries = bool_frame((sma_short.gt(sma_long)) & close.gt(sma_short))
sma_exits = bool_frame((sma_short.lt(sma_long)) | close.lt(sma_long))
pf_sma = vbt.Portfolio.from_signals(
    close=close,
    entries=sma_entries,
    exits=sma_exits,
    size=float_frame(position_size.where(sma_entries, 0.0), 0.0),
    size_type="amount",
    init_cash=INITIAL_CAPITAL,
    fees=FEES,
    slippage=SLIPPAGE,
    freq=VECTORBT_FREQ,
    group_by=True,
    cash_sharing=True,
    call_seq="auto",
)

buy_hold_returns = returns.reindex(close.index).fillna(0.0).dot(weights)
buy_hold_value = INITIAL_CAPITAL * (1.0 + buy_hold_returns).cumprod()

log_step(
    "backtest_ready",
    combined_trade_count=pf_gap.trades.count(),
    common_trade_count=pf_common.trades.count(),
    long_trend_trade_count=pf_long_trend.trades.count(),
    exhaustion_short_trade_count=pf_exhaustion_short.trades.count(),
    final_gap_value=round(float(pf_gap.value().iloc[-1]), 2),
)
pf_gap.stats()



[backtest_ready]
  combined_trade_count: 109
  common_trade_count: 151
  long_trend_trade_count: 26
  exhaustion_short_trade_count: 0
  final_gap_value: 98906.01


Start                             2025-07-14 00:00:00
End                               2026-07-13 00:00:00
Period                              257 days 00:00:00
Start Value                               100000.0000
End Value                                  98906.0076
Total Return [%]                              -1.0940
Benchmark Return [%]                          60.0962
Max Gross Exposure [%]                         2.5895
Total Fees Paid                               50.3289
Max Drawdown [%]                               1.1240
Max Drawdown Duration               256 days 00:00:00
Total Trades                                      109
Total Closed Trades                               109
Total Open Trades                                   0
Open Trade PnL                                 0.0000
Win Rate [%]                                  34.8624
Best Trade [%]                                45.8724
Worst Trade [%]                              -21.2794
Avg Winning Trade [%]       

## 10. Entry, Exit, And Common Gap Fill Visualization

Bu grafikler backtest sinyallerini ve common gap hedefini aynı mum üzerinde gösterir. Common Up için fill hedefi önceki günün high seviyesi, Common Down için fill hedefi önceki günün low seviyesidir.


In [266]:
ENTRY_EXIT_PLOT_WINDOW = 500

trades_for_plot = pf_gap.trades.records_readable.copy()
if not trades_for_plot.empty:
    trades_for_plot.columns = [str(column) for column in trades_for_plot.columns]

    def normalize_trade_column(value):
        if value in close.columns:
            return value
        try:
            return close.columns[int(value)]
        except (TypeError, ValueError, IndexError):
            return value

    if "Column" in trades_for_plot.columns:
        trades_for_plot["_symbol"] = trades_for_plot["Column"].map(normalize_trade_column)
    else:
        trades_for_plot["_symbol"] = None

    for timestamp_column in ["Entry Timestamp", "Exit Timestamp"]:
        if timestamp_column in trades_for_plot.columns:
            trades_for_plot[timestamp_column] = pd.to_datetime(trades_for_plot[timestamp_column])
else:
    trades_for_plot["_symbol"] = []

def first_trade_value(row: pd.Series, names: list[str]):
    for name in names:
        if name in row and pd.notna(row[name]):
            return row[name]
    return np.nan

log_step("entry_exit_plot_selection", plot_window=ENTRY_EXIT_PLOT_WINDOW, plotted_symbols=PLOT_SYMBOLS)

for symbol in PLOT_SYMBOLS:
    idx = close.index[-ENTRY_EXIT_PLOT_WINDOW:]
    sample = pd.DataFrame(
        {
            "open": open_[symbol],
            "high": high[symbol],
            "low": low[symbol],
            "close": close[symbol],
            "common_up": common_up[symbol],
            "common_down": common_down[symbol],
            "common_up_target_hit": common_up_target_hit[symbol],
            "common_down_target_hit": common_down_target_hit[symbol],
            "common_fill_target": common_fill_target[symbol],
            "common_target_hit_price": common_target_hit_price[symbol],
        }
    ).loc[idx]

    fig = go.Figure()
    fig.add_trace(
        go.Candlestick(
            x=sample.index,
            open=sample["open"],
            high=sample["high"],
            low=sample["low"],
            close=sample["close"],
            name=symbol,
        )
    )

    symbol_trades = trades_for_plot[trades_for_plot["_symbol"].eq(symbol)].copy()
    if not symbol_trades.empty and "Entry Timestamp" in symbol_trades.columns:
        symbol_trades = symbol_trades[
            symbol_trades["Entry Timestamp"].between(sample.index.min(), sample.index.max())
            | symbol_trades.get("Exit Timestamp", symbol_trades["Entry Timestamp"]).between(sample.index.min(), sample.index.max())
        ]

    for _, trade in symbol_trades.iterrows():
        direction = str(trade.get("Direction", "")).lower()
        is_short_trade = "short" in direction
        entry_ts = trade.get("Entry Timestamp")
        exit_ts = trade.get("Exit Timestamp")
        entry_price = first_trade_value(trade, ["Avg Entry Price", "Entry Price"])
        exit_price = first_trade_value(trade, ["Avg Exit Price", "Exit Price"])

        if pd.notna(entry_ts) and sample.index.min() <= entry_ts <= sample.index.max():
            fig.add_trace(
                go.Scatter(
                    x=[entry_ts],
                    y=[entry_price],
                    mode="markers",
                    marker=dict(
                        color="#dc2626" if is_short_trade else "#16a34a",
                        symbol="triangle-down" if is_short_trade else "triangle-up",
                        size=14,
                        line=dict(width=1, color="#111827"),
                    ),
                    name="Executed Short Entry" if is_short_trade else "Executed Long Entry",
                    showlegend=False,
                )
            )

        if pd.notna(exit_ts) and sample.index.min() <= exit_ts <= sample.index.max():
            fig.add_trace(
                go.Scatter(
                    x=[exit_ts],
                    y=[exit_price],
                    mode="markers",
                    marker=dict(
                        color="#f97316" if is_short_trade else "#2563eb",
                        symbol="x",
                        size=12,
                        line=dict(width=1, color="#111827"),
                    ),
                    name="Executed Short Exit" if is_short_trade else "Executed Long Exit",
                    showlegend=False,
                )
            )

    common_points = sample[sample["common_up"] | sample["common_down"]]
    if not common_points.empty:
        fig.add_trace(
            go.Scatter(
                x=common_points.index,
                y=common_points["common_fill_target"],
                mode="markers",
                marker=dict(color="#7c3aed", symbol="diamond", size=10),
                name="Entry Fill Target",
                text=np.where(
                    common_points["common_up"],
                    "Common Up short target: previous bar high",
                    "Common Down long target: previous bar low",
                ),
            )
        )

    hit_points = sample[sample["common_up_target_hit"] | sample["common_down_target_hit"]]
    if not hit_points.empty:
        fig.add_trace(
            go.Scatter(
                x=hit_points.index,
                y=hit_points["common_target_hit_price"],
                mode="markers",
                marker=dict(color="#facc15", symbol="star", size=15, line=dict(width=1, color="#111827")),
                name="Target Hit After Entry",
            )
        )

    legend_traces = [
        ("Executed Long Entry", "#16a34a", "triangle-up"),
        ("Executed Long Exit", "#2563eb", "x"),
        ("Executed Short Entry", "#dc2626", "triangle-down"),
        ("Executed Short Exit", "#f97316", "x"),
    ]
    for name, color, marker_symbol in legend_traces:
        fig.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                marker=dict(color=color, symbol=marker_symbol, size=11, line=dict(width=1, color="#111827")),
                name=name,
            )
        )

    fig.update_layout(
        title=f"{symbol} Executed Entries / Exits / Common Gap Fill Targets",
        xaxis_title="Date",
        yaxis_title="Price",
        xaxis_rangeslider_visible=False,
        height=720,
    )
    fig.show()



[entry_exit_plot_selection]
  plot_window: 500
  plotted_symbols: ['TURSG.IS', 'HLGYO.IS', 'ENTRA.IS', 'OSTIM.IS', 'GOODY.IS']


In [267]:
def stats_from_value_curve(name: str, value: pd.Series, split_date: pd.Timestamp | None = None) -> dict[str, float | str]:
    value = value.dropna()
    bar_returns = value.pct_change().dropna()
    observed_bars_per_day = value.groupby(value.index.date).size().median() if len(value) else np.nan
    periods_per_year = 252 * observed_bars_per_day if pd.notna(observed_bars_per_day) and observed_bars_per_day > 0 else 252
    if split_date is not None:
        test_value = value.loc[value.index >= split_date]
        test_returns = test_value.pct_change().dropna()
    else:
        test_value = value
        test_returns = bar_returns
    total_return = value.iloc[-1] / value.iloc[0] - 1.0 if len(value) > 1 else np.nan
    test_return = test_value.iloc[-1] / test_value.iloc[0] - 1.0 if len(test_value) > 1 else np.nan
    sharpe = np.sqrt(periods_per_year) * bar_returns.mean() / bar_returns.std() if bar_returns.std() > 0 else np.nan
    test_sharpe = np.sqrt(periods_per_year) * test_returns.mean() / test_returns.std() if test_returns.std() > 0 else np.nan
    drawdown = value / value.cummax() - 1.0
    return {
        "strategy": name,
        "final_value_tl": value.iloc[-1],
        "total_return": total_return,
        "max_drawdown": drawdown.min(),
        "annualized_sharpe": sharpe,
        "test_return": test_return,
        "test_annualized_sharpe": test_sharpe,
        "observed_bars_per_day": observed_bars_per_day,
    }

value_curves = pd.DataFrame(
    {
        "gap_combined": pf_gap.value(),
        "long_breakaway_runaway": pf_long_trend.value(),
        "common_gap_fill": pf_common.value(),
        "exhaustion_short": pf_exhaustion_short.value(),
        "sma_baseline": pf_sma.value(),
        "optimized_buy_hold": buy_hold_value.reindex(close.index).ffill(),
    }
).dropna(how="all")

stats_table = pd.DataFrame(
    [stats_from_value_curve(column, value_curves[column], train_end) for column in value_curves.columns]
).sort_values("test_return", ascending=False)

display(stats_table)
log_step(
    "strategy_stats_summary",
    best_test_strategy=stats_table.iloc[0]["strategy"] if not stats_table.empty else None,
    best_test_return=round(float(stats_table.iloc[0]["test_return"]), 4) if not stats_table.empty else None,
    final_values=value_curves.iloc[-1].round(2).to_dict(),
)
strategy_value_curves = value_curves.drop(columns=["optimized_buy_hold"])
px.line(
    strategy_value_curves,
    title="Strategy Equity Curves",
    labels={"value": "Portfolio Value (TL)", "index": "Date"},
).show()
px.line(
    value_curves[["optimized_buy_hold"]],
    title="Optimized Buy & Hold Equity Curve",
    labels={"value": "Portfolio Value (TL)", "index": "Date"},
).show()


,strategy,final_value_tl,total_return,max_drawdown,annualized_sharpe,test_return,test_annualized_sharpe,observed_bars_per_day
5,optimized_buy_hold,506210.6223,4.0621,-0.0384,12.5060,0.3282,5.8814,1.0000
4,sma_baseline,105601.1713,0.0560,-0.0038,4.9501,0.0350,6.9681,1.0000
1,long_breakaway_runaway,100409.8307,0.0041,-0.0007,2.5775,0.0014,2.2247,1.0000
0,gap_combined,98906.0076,-0.0109,-0.0112,-3.6515,0.0000,0.0041,1.0000
3,exhaustion_short,100000.0000,0.0000,0.0000,NaN,0.0000,NaN,1.0000
2,common_gap_fill,94452.3542,-0.0555,-0.0555,-4.7176,-0.0226,-6.5226,1.0000



[strategy_stats_summary]
  best_test_strategy: optimized_buy_hold
  best_test_return: 0.3282
  final_values: {'gap_combined': 98906.01, 'long_breakaway_runaway': 100409.83, 'common_gap_fill': 94452.35, 'exhaustion_short': 100000.0, 'sma_baseline': 105601.17, 'optimized_buy_hold': 506210.62}


## 10. Trades, Validation, And Diagnostics

In [268]:
trades = pf_gap.trades.records_readable
if len(trades) > 0:
    display(trades.tail(30))
else:
    print("No trades were generated by the combined gap strategy.")

signal_validation = pd.DataFrame(
    {
        "signal": [
            "trend_long_breakaway_runaway",
            "common_down_fill_long",
            "common_up_fill_short",
            "common_up_target_hits_after_entry",
            "common_down_target_hits_after_entry",
            "exhaustion_up_short",
            "trend_down_short",
        ],
        "events": [
            int(trend_long_entries.sum().sum()),
            int(common_down_fill_long.sum().sum()),
            int(common_up_fill_short.sum().sum()),
            int(common_up_target_hit.sum().sum()),
            int(common_down_target_hit.sum().sum()),
            int(exhaustion_short_entries.sum().sum()),
            int(trend_short_entries.sum().sum()),
        ],
    }
)
display(signal_validation)
log_step(
    "trade_diagnostics",
    combined_trade_count=len(trades),
    signal_validation=signal_validation.to_dict("records"),
)

monthly_returns = value_curves["gap_combined"].pct_change().resample("ME").apply(lambda values: (1 + values).prod() - 1)
px.bar(monthly_returns, title="Combined Gap Strategy Monthly Returns", labels={"value": "Monthly Return"}).show()


,Exit Trade Id,Column,Size,Entry Timestamp,Avg Entry Price,Entry Fees,Exit Timestamp,Avg Exit Price,Exit Fees,PnL,Return,Direction,Status,Position Id
79,79,MEGMT.IS,1.0000,2026-04-27,83.6336,0.0836,2026-05-20,81.5184,0.0815,-2.2803,-0.0273,Long,Closed,79
80,80,MEPET.IS,6.0000,2025-09-24,15.4254,0.0926,2025-10-13,15.9540,0.0957,2.9834,0.0322,Long,Closed,80
81,81,NETCD.IS,1.0000,2026-02-06,55.5944,0.0556,2026-02-09,61.2000,0.0612,-5.7224,-0.1029,Short,Closed,81
82,82,NETCD.IS,1.0000,2026-02-10,67.2327,0.0672,2026-02-11,74.0000,0.0740,-6.9085,-0.1028,Short,Closed,82
83,83,NETCD.IS,1.0000,2026-02-12,81.3186,0.0813,2026-02-13,89.5000,0.0895,-8.3522,-0.1027,Short,Closed,83
84,84,NETCD.IS,1.0000,2026-02-16,98.3515,0.0984,2026-02-18,108.5000,0.1085,-10.3553,-0.1053,Short,Closed,84
85,85,PASEU.IS,2.0000,2025-12-22,147.2471,0.2945,2025-12-24,140.6592,0.2813,-13.7516,-0.0467,Long,Closed,85
86,86,PEKGY.IS,37.0000,2025-07-16,2.2078,0.0817,2025-07-17,2.4300,0.0899,-8.3934,-0.1027,Short,Closed,86
87,87,PEKGY.IS,31.0000,2025-07-18,2.6673,0.0827,2025-07-21,2.9300,0.0908,-8.3163,-0.1006,Short,Closed,87
88,88,PEKGY.IS,26.0000,2025-07-22,3.2168,0.0836,2025-07-25,3.7900,0.0985,-15.0859,-0.1804,Short,Closed,88


,signal,events
0,trend_long_breakaway_runaway,545
1,common_down_fill_long,1712
2,common_up_fill_short,1997
3,common_up_target_hits_after_entry,1012
4,common_down_target_hits_after_entry,937
5,exhaustion_up_short,213
6,trend_down_short,232



[trade_diagnostics]
  combined_trade_count: 109
  signal_validation: [{'signal': 'trend_long_breakaway_runaway', 'events': 545}, {'signal': 'common_down_fill_long', 'events': 1712}, {'signal': 'common_up_fill_short', 'events': 1997}, {'signal': 'common_up_target_hits_after_entry', 'events': 1012}, {'signal': 'common_down_target_hits_after_entry', 'events': 937}, {'signal': 'exhaustion_up_short', 'events': 213}, {'signal': 'trend_down_short', 'events': 232}]
